In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv("cajun.csv")

print(df.head())
print(df.info())

                                                  id  ...                                        ingredients
0                                  Jambalaya-1370876  ...  [olive oil, celery, white onion, red bell pepp...
1       Easy-Spicy-Cajun-Shrimp-_with-sauce_-1373106  ...  [jumbo shrimp, garlic cloves, cajun seasoning,...
2                              Shrimp-Creole-1365578  ...  [olive oil, onions, celery, green bell pepper,...
3  Fully-Loaded-Jambalaya-with-Andouille-Sausage-...  ...  [Uncle Ben's® Ready Rice® Original Long Grain,...
4                            Cajun-Jambalaya-1360690  ...  [shrimp, boneless skinless chicken breasts, tu...

[5 rows x 7 columns]
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  500 non-null    str    
 1   recipeName          500 non-null    str    
 2   rating              500 non-nu

In [3]:
df = df.dropna()  # simple baseline cleaning

In [4]:
TARGET_COLUMN = "target"   # <-- change this
TEXT_COLUMN = "text"       # <-- change this if NLP column exists

In [7]:
for c in df.columns:
    print(repr(c))

'id'
'recipeName'
'rating'
'totalTimeInSeconds'
'course'
'cuisine'
'ingredients'


In [8]:
[col for col in df.columns if "target" in col.lower()]

[]

In [13]:
print(globals().keys())

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__session__', '_i', '_ii', '_iii', '_i1', 'pd', 'np', 'train_test_split', 'RandomForestClassifier', 'ExtraTreesClassifier', 'accuracy_score', 'classification_report', 'KMeans', 'StandardScaler', 'TfidfVectorizer', 'Pipeline', 'ColumnTransformer', '_i2', 'df', '_i3', '_i4', 'TARGET_COLUMN', 'TEXT_COLUMN', '_i5', '_i6', '_i7', 'c', '_i8', '_8', '_i9', '_i10', '_i11', '_i12', '_i13'])


In [17]:
print(df.columns)
df.head()

Index(['id', 'recipeName', 'rating', 'totalTimeInSeconds', 'course', 'cuisine',
       'ingredients'],
      dtype='str')


,id,recipeName,rating,totalTimeInSeconds,course,cuisine,ingredients
0,Jambalaya-1370876,Jambalaya,4,3300.0,[Main Dishes],[Cajun & Creole],"[olive oil, celery, white onion, red bell pepp..."
1,Easy-Spicy-Cajun-Shrimp-_with-sauce_-1373106,Easy Spicy Cajun Shrimp (with sauce),4,1800.0,[Main Dishes],[Cajun & Creole],"[jumbo shrimp, garlic cloves, cajun seasoning,..."
2,Shrimp-Creole-1365578,Shrimp Creole,4,4200.0,[Main Dishes],[Cajun & Creole],"[olive oil, onions, celery, green bell pepper,..."
3,Fully-Loaded-Jambalaya-with-Andouille-Sausage-...,Fully Loaded Jambalaya with Andouille Sausage,3,1200.0,[Main Dishes],[Cajun & Creole],"[Uncle Ben's® Ready Rice® Original Long Grain,..."
4,Cajun-Jambalaya-1360690,Cajun Jambalaya,3,2700.0,[Main Dishes],[Cajun & Creole],"[shrimp, boneless skinless chicken breasts, tu..."


In [19]:
X = df.drop(columns=["Cluster"], errors="ignore")

In [20]:
if TEXT_COLUMN in df.columns:
    vectorizer = TfidfVectorizer(stop_words='english', max_features=20)
    X_text = vectorizer.fit_transform(df[TEXT_COLUMN].astype(str))

    print(vectorizer.get_feature_names_out())